##Imports

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import timedelta
import sys
sys.path.append("../aws_utils")
import aws_utils

In [0]:
env = aws_utils.get_env_prefix()
#env = 'p' #--change here!!! when we go to prod it will be one env. We cannot run for the main tables in dev because it is not in synch. Only used for GETTING data from there we do not write on tables.
env2 = aws_utils.get_env_prefix() # same as env - previously was separate for export tables

## Date

In [0]:
export_date_df = spark.sql("""
    SELECT last_day(add_months(cast(getdate() as date), -2)) AS export_date
""")

export_date = export_date_df.collect()[0]['export_date']
print(f"Calculated export_date: {export_date}")

In [0]:
print("Checking data in old_style_rejections against final rejections table...")

date1_result = spark.sql(f"SELECT max(export_date) FROM `{env2}-cdh-delivery`.tiresias.old_style_rejections").collect()

date2 = spark.sql(f"select MAX(export_date) from `{env2}-cdh-delivery`.tiresias.rejections").collect()[0][0]

# Print date2 (Max date from destination table)
print(f"Max date from 'rejections' (date2): {date2}")

#CHECKS
# old_style_rejections is empty
if not date1_result:
    print("No data found in 'old_style_rejections'. No new rejection files were loaded in the previous step.")
    print("Skipping all remaining steps.")
    dbutils.notebook.exit("Skipping: No new rejection data to process.")
else:
    date1 = date1_result[0][0]
    
    # Print date1 (New data from source table)
    print(f"Max date from 'old_style_rejections' (date1): {date1}")
    print(f"--- Comparing: date1={date1} | date2={date2} ---")

    # Destination table is empty (first run)
    if date2 is None:
        print(f"Action: 'rejections' table is empty. Continuing to load new data for {date1}.")
    
    # Data already processed
    elif date1 == date2:
        print(f"Action: Data for {date1} already exists in 'rejections'. Skipping.")
        dbutils.notebook.exit(f"Skipping: Data for {date1} already processed.")

    # Source data is old/outdated 
    elif date1 < date2:
        print(f"Info: Source data ({date1}) is older than existing max data ({date2}). Checking if {date1} exists...")
        
        # Check if this specific old date already exists
        check_old_date_exists = spark.sql(f"SELECT 1 FROM `{env2}-cdh-delivery`.tiresias.rejections WHERE export_date = '{date1}' LIMIT 1").collect()
        
        if check_old_date_exists:
            # It's old AND it already exists. Skip.
            print(f"Action: This old data ({date1}) has already been processed. Skipping.")
            dbutils.notebook.exit(f"Skipping: Stale data ({date1}) already processed.")
        else:
            # It's old, but it DOES NOT exist. Allow it to run.
            print(f"Action: This old data ({date1}) was not found in 'rejections'. Allowing run to fill missing data.")
    
    # Correct new data
    elif date1 > date2:
        print(f"Action: New data ({date1}) found. Continuing execution.")

## Insert Rejections

In [0]:
insert_into_rejections = spark.sql(f"""
INSERT INTO `{env2}-cdh-delivery`.tiresias.rejections 
    (export_date, rejection_file, account_number, Customer_1, Customer_2, error_code, export_type)
SELECT 
    a.export_date,  
    rejection_file,
    ltrim(rtrim(account_number)),
    ltrim(rtrim(Customer_1)),
    ltrim(rtrim(Customer_2)),
    ltrim(rtrim(error_code)), 
    export_type
FROM 
    `{env2}-cdh-delivery`.tiresias.old_style_rejections a
WHERE export_date = '{export_date}'
    AND ltrim(rtrim(COALESCE(error_code, ''))) <> ''
""")

display(insert_into_rejections)

## Update History Tables

In [0]:
merge_business_history = spark.sql(f"""
MERGE INTO `{env2}-cdh-delivery`.tiresias.export_business_hist t
USING (
    SELECT
        h.cdi,
        h.`ΚΩΔΙΚΟΣ_ΛΟΓΑΡΙΑΣΜΟΥ_ΧΟΡΗΓΗΣΗΣ`,
        h.portfolio,
        h.`ΑΦΜ`,
        h.`ΑΔΤ`,
        h.rpt_date,
        1 AS has_error,
        j.errors
    FROM `{env2}-cdh-delivery`.tiresias.export_business_hist h
    INNER JOIN (
        SELECT
            export_date,
            account_number,
            Customer_1,
            Customer_2,
            STRING_AGG(error_code, ' | ') AS errors
        FROM `{env2}-cdh-delivery`.tiresias.rejections
        WHERE export_date = '{export_date}'
        GROUP BY export_date, account_number, Customer_1, Customer_2
    ) j ON j.account_number = h.`ΚΩΔΙΚΟΣ_ΛΟΓΑΡΙΑΣΜΟΥ_ΧΟΡΗΓΗΣΗΣ`
    AND j.Customer_1 = COALESCE(h.`ΑΦΜ`, '')
    AND j.Customer_2 = COALESCE(h.`ΑΔΤ`, '')
    AND j.export_date = h.rpt_date
) s
ON (
    s.cdi = t.cdi
    AND s.`ΚΩΔΙΚΟΣ_ΛΟΓΑΡΙΑΣΜΟΥ_ΧΟΡΗΓΗΣΗΣ` = t.`ΚΩΔΙΚΟΣ_ΛΟΓΑΡΙΑΣΜΟΥ_ΧΟΡΗΓΗΣΗΣ`
    AND s.portfolio = t.portfolio
    AND s.rpt_date = t.rpt_date
)
WHEN MATCHED
THEN
    UPDATE SET
        t.has_error = s.has_error,
        t.errors = s.errors
""")

display(merge_business_history) 

In [0]:
merge_consumer_history = spark.sql(f"""
MERGE INTO `{env2}-cdh-delivery`.tiresias.export_consumer_hist t
USING (
    SELECT
        h.cdi,
        h.`ΚΩΔΙΚΟΣ_ΛΟΓΑΡΙΑΣΜΟΥ_ΧΟΡΗΓΗΣΗΣ`,
        h.portfolio,
        h.`ΑΦΜ`,
        h.`ΑΔΤ`,
        h.rpt_date,
        1 AS has_error,
        j.errors
    FROM `{env2}-cdh-delivery`.tiresias.export_consumer_hist h
    INNER JOIN (
        SELECT
            export_date,
            account_number,
            Customer_1,
            Customer_2,
            STRING_AGG(error_code, ' | ') AS errors
        FROM `{env2}-cdh-delivery`.tiresias.rejections
        WHERE export_date = '{export_date}'
        GROUP BY export_date, account_number, Customer_1, Customer_2
    ) j ON j.account_number = h.`ΚΩΔΙΚΟΣ_ΛΟΓΑΡΙΑΣΜΟΥ_ΧΟΡΗΓΗΣΗΣ`
    AND j.Customer_1 = COALESCE(h.`ΑΦΜ`, '')
    AND j.Customer_2 = COALESCE(h.`ΑΔΤ`, '')
    AND j.export_date = h.rpt_date
) s
ON (
    s.cdi = t.cdi
    AND s.`ΚΩΔΙΚΟΣ_ΛΟΓΑΡΙΑΣΜΟΥ_ΧΟΡΗΓΗΣΗΣ` = t.`ΚΩΔΙΚΟΣ_ΛΟΓΑΡΙΑΣΜΟΥ_ΧΟΡΗΓΗΣΗΣ`
    AND s.portfolio = t.portfolio
    AND s.rpt_date = t.rpt_date
)
WHEN MATCHED
THEN
    UPDATE SET
        t.has_error = s.has_error,
        t.errors = s.errors
""")

display(merge_consumer_history)